In [2]:
using ITensors
using ITensorMPS
using Random

using LinearAlgebra
using Statistics
#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end 


function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)

    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end
#=
    Returns the density of up and down 
    electrons.
=#
function density_operators(N, psi)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
    orthogonalize!(psi, j)
    psidag_j = dag(prime(psi[j], "Site"))
    upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
    dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
    updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

function Ep(rdm, L)
    println("trace (rho) = ", tr(rdm))
    println("ishermitian(rho) = ", ishermitian(rdm))
    lambdas = eigvals(rdm)
    S = 0
    println("eigenvals = ", lambdas)
    for lambda in lambdas
        lambda = real(lambda) 
        # S -= lambda * log2(lambda)
        # CRITICAL: Handle lambda <= 0 for log2
        if lambda > 1e-15 # A small threshold to avoid errors with log2
            S -= lambda * log2(lambda)
        end
    end 
    return S 
end

function build_1_particle_rdm(psi) 
    L = length(psi)
    #=  
        N and not L since any site can have spin up or down
    =#
    rho_1 = zeros(ComplexF64, 2*L, 2*L) 

    Cupup = correlation_matrix(psi, "Cdagup", "Cup")
    Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
    Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
    Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")   

    for i in 1:L
        for j in 1:L
            # Blocks of the correlation matrix.
            i_up = 2 * (i-1) + 1 
            i_dn = 2 * (i-1) + 2
            j_up = 2 * (j-1) + 1
            j_dn = 2 * (j-1) + 2

            rho_1[i_up, j_up] = Cupup[i,j]
            rho_1[i_up, j_dn] = Cupdn[i,j]
            rho_1[i_dn, j_up] = Cdnup[i,j]
            rho_1[i_dn, j_dn] = Cdndn[i,j]
        end 
    end
    rho_1 = rho_1 / L
end

build_1_particle_rdm (generic function with 1 method)

In [32]:
L = 7
sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 10

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

println(state)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.5 
V = -7.5
J = 1.0

H = H_EHM(L, J, U, V, sites)

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

["Up", "Dn", "Up", "Dn", "Up", "UpDn", "Emp"]
After sweep 1 energy=-73.9130815863746  maxlinkdim=50 maxerr=2.85E-10 time=0.115
After sweep 2 energy=-73.91489442526986  maxlinkdim=55 maxerr=7.48E-15 time=0.132
After sweep 3 energy=-73.9149394723613  maxlinkdim=54 maxerr=9.98E-15 time=0.127
After sweep 4 energy=-73.91498160899782  maxlinkdim=54 maxerr=9.89E-15 time=0.138
After sweep 5 energy=-73.91502261058709  maxlinkdim=54 maxerr=9.43E-15 time=0.155
After sweep 6 energy=-73.91506991231796  maxlinkdim=55 maxerr=8.74E-15 time=0.159
After sweep 7 energy=-73.91511846675324  maxlinkdim=55 maxerr=7.91E-15 time=0.175
After sweep 8 energy=-73.91516289350318  maxlinkdim=56 maxerr=6.46E-15 time=0.153
After sweep 9 energy=-73.91520617674219  maxlinkdim=55 maxerr=8.57E-15 time=0.166
After sweep 10 energy=-73.91524940243566  maxlinkdim=56 maxerr=9.23E-15 time=0.156


(-73.91524940243566, MPS
[1] ((dim=4|id=430|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=421|"Link,l=1") <Out>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 1
 3: QN(("Nf",6,-1),("Sz",2)) => 1
 4: QN(("Nf",7,-1),("Sz",1)) => 1)
[2] ((dim=16|id=594|"Link,l=2") <Out>
 1: QN(("Nf",3,-1),("Sz",1)) => 1
 2: QN(("Nf",4,-1),("Sz",0)) => 2
 3: QN(("Nf",4,-1),("Sz",2)) => 2
 4: QN(("Nf",5,-1),("Sz",-1)) => 1
 5: QN(("Nf",5,-1),("Sz",1)) => 4
 6: QN(("Nf",5,-1),("Sz",3)) => 1
 7: QN(("Nf",6,-1),("Sz",0)) => 2
 8: QN(("Nf",6,-1),("Sz",2)) => 2
 9: QN(("Nf",7,-1),("Sz",1)) => 1, (dim=4|id=649|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=421|"Link,l=1") <In>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 

In [9]:
rho_1 = build_1_particle_rdm(psi)

14×14 Matrix{ComplexF64}:
  0.000618109+0.0im           0.0+0.0im  …          0.0+0.0im
          0.0+0.0im    2.43613e-5+0.0im      1.63913e-5+0.0im
   0.00893506+0.0im           0.0+0.0im             0.0+0.0im
          0.0+0.0im   0.000203135+0.0im     0.000123629+0.0im
 -0.000516457+0.0im           0.0+0.0im             0.0+0.0im
          0.0+0.0im   0.000712959+0.0im  …   8.20257e-5+0.0im
   4.79007e-5+0.0im           0.0+0.0im             0.0+0.0im
          0.0+0.0im  -0.000173549+0.0im     -9.43339e-5+0.0im
 -0.000360737+0.0im           0.0+0.0im             0.0+0.0im
          0.0+0.0im    0.00160593+0.0im      0.00128212+0.0im
   0.00249719+0.0im           0.0+0.0im  …          0.0+0.0im
          0.0+0.0im   0.000233669+0.0im     0.000181382+0.0im
  0.000170674+0.0im           0.0+0.0im             0.0+0.0im
          0.0+0.0im    1.63913e-5+0.0im      1.27312e-5+0.0im

In [10]:
E_p = Ep(rho_1, L) - log2(L)
println("E_p = ", E_p)

trace (rho) = 1.0000000000000002 + 0.0im
ishermitian(rho) = true
eigenvals = [5.2442274923991716e-8, 2.277983621802567e-7, 6.298080121335313e-7, 6.097756721626779e-6, 8.478399534337703e-6, 0.00022578390627599554, 0.00022781594191284587, 0.14263215542812438, 0.14263914156537455, 0.1428411095070441, 0.14284778487014468, 0.1428566112089996, 0.14285698355973833, 0.1428571278074803]
E_p = 0.005118508643301656


In [17]:
  N = 8
  m = 4

  s = siteinds("Electron", N; conserve_qns=true)
  psi = random_mps(s, n -> isodd(n) ? "Up" : "Dn"; linkdims=m)
  
  Cuu = correlation_matrix(psi, "Cdagup", "Cup")

8×8 Matrix{Float64}:
  0.16083       0.250937      0.0642502   …  -0.000328625  -0.00106088
  0.250937      0.401214      0.0783951      -0.000487501  -0.00154875
  0.0642502     0.0783951     0.632831       -0.00227759   -0.00257705
  0.0491944     0.0705498    -0.12255        -0.00178112   -0.00401016
 -0.0130343    -0.0190606    -0.0377959       0.00533961    0.00497998
 -0.018253     -0.0265232    -0.0207015   …  -0.0272933    -0.0366269
 -0.000328625  -0.000487501  -0.00227759      0.643955      0.215661
 -0.00106088   -0.00154875   -0.00257705      0.215661      0.0991394

- trying to create a non-uniform mesh grid to compute the phase diagram.

In [15]:
# param1_segment1 = range(0.0, stop=0.9, length=10)
# param1_segment2 = range(0.91, stop=1.09, length=40)
# param1_segment3 = range(1.1, stop=2.0, length=10)
# param1_values = vcat(collect(param1_segment1), collect(param1_segment2), collect(param1_segment3))
# param2_values = collect(range(0.0, stop=1.0, length=50))

In [30]:
N = 2
m = 4

s = siteinds("Electron", N; conserve_qns=true)

psi = productMPS(s, ["Up", "Dn"])

MPS
[1] ((dim=4|id=799|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=1|id=136|"Link,l=1") <In>
 1: QN(("Nf",1,-1),("Sz",1)) => 1)
[2] ((dim=1|id=136|"Link,l=1") <Out>
 1: QN(("Nf",1,-1),("Sz",1)) => 1, (dim=4|id=632|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1)


In [19]:
rho_1 = build_1_particle_rdm(psi)

println(tr(rho_1))
println("is hermitian ? ", ishermitian(rho_1))

1.0 + 0.0im
is hermitian ? true


In [20]:
real(rho_1)

4×4 Matrix{Float64}:
 0.5  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.5

In [22]:
Cupup = correlation_matrix(psi, "Cdagup", "Cup")
Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")

println("Cupup = ", Cupup)
println("Cupdn = ", Cupdn)
println("Cdnup = ", Cdnup)
println("Cdndn = ", Cdndn)

Cupup = [1.0 0.0; 0.0 0.0]
Cupdn = [0.0 0.0; 0.0 0.0]
Cdnup = [0.0 0.0; 0.0 0.0]
Cdndn = [0.0 0.0; 0.0 1.0]


In [23]:
E_p = Ep(rho_1, N) - log2(N)
println("E_p = ", E_p)

trace (rho) = 1.0 + 0.0im
ishermitian(rho) = true
eigenvals = [0.0, 0.0, 0.5, 0.5]
E_p = 0.0


In [33]:
function density_operators(N, psi, sites)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
        orthogonalize!(psi, j)
        psidag_j = dag(prime(psi[j], "Site"))
        upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
        dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
        updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

upd, dnd, updn = density_operators(N, psi, sites)

([0.00043789457729379245, 0.08856280828242392], [4.448690794936952e-5, 0.0026433171076354228], [2.1033690360340057e-7, 0.0025298538215040504])

In [70]:
magnetization = (upd .- dnd) / 2
print(magnetization)

[0.00019670383467221145, 0.04295974558739425]

In [71]:
charge_density = (upd .+ dnd)
print(charge_density)

[0.000482381485243162, 0.09120612539005934]

In [76]:
function m_sdw(L, Sj)
    m_sdw = 0.0 
    for j in 1:L 
        println(j, " ", Sj[j])
        m_sdw += (-1)^(j) * Sj[j]
    end
    return m_sdw / L
end
function m_cdw(L, nj)
    m_cdw = 0.0
    for j in 1:L 
        println(j, " ", nj[j])
        m_cdw += (-1)^(j) * (nj[j] - 1)
    end
    return m_cdw / L 
end 

m_cdw (generic function with 1 method)

In [77]:
println("|m_sdw| = ", abs(m_sdw(N, magnetization)))
println("|m_cdw| = ", abs(m_cdw(N, charge_density)))

1 0.00019670383467221145
2 0.04295974558739425
|m_sdw| = 0.02138152087636102
1 0.000482381485243162
2 0.09120612539005934
|m_cdw| = 0.045361871952408095


2-rdm

In [27]:
using ITensors, ITensorMPS

N = 10
sites = siteinds("Electron", N)

os = OpSum()

os += "Cdagup", 2, "Cdagup", 4, "Cdn", 3, "Cdn", 1

# Again, you can add more terms and coefficients as needed.

my_operator_MPO = MPO(os, sites)

println("\nCreated MPO for the spinful operator:")
println(os)


Created MPO for the spinful operator:
sum(
  1.0 Cdagup(2,) Cdagup(4,) Cdn(3,) Cdn(1,)
)


In [ ]:
spins = ["up", "dn"]